### XGBoost — formulas, derived

**Gradient and Hessian for log-loss** (z = raw score before sigmoid)
g = dL/dz = p - y                            (derived in notebook 1)
h = d^2L/dz^2 = d(p-y)/dz = dp/dz = p(1-p)    (differentiate g again w.r.t. z -- same sigmoid derivative as before, now the curvature term instead of canceling out)

**Newton's step -- why g and h together, not just g**
Optimal per-point correction = -g/h = -(p-y) / [p(1-p)]
Uses curvature (h) to size the correction directly, instead of guessing a step and hoping (which is what a manually-tuned learning rate compensates for in plain gradient boosting).

**Leaf value** (aggregated over every point that lands in a leaf)
leaf_value = -sum(g) / (sum(h) + lambda)
lambda dampens the value for leaves with few points (small sum(h)), preventing an overfit, extreme correction from a leaf with only a handful of points.

**Leaf score** (replaces Gini -- scores how good one group is)
leaf_score = (sum(g))^2 / (sum(h) + lambda)
Bigger = this group has a large, consistent correction to make = worth carving into its own leaf.

**Gain** (replaces weighted_gini -- scores a candidate split)
gain = leaf_score(left) + leaf_score(right) - leaf_score(parent)
(Real XGBoost also applies a 1/2 factor and subtracts a fixed penalty gamma per split; the core comparison logic here is the same.) Best split = highest gain. Gain <= 0 means splitting doesn't help -- that's also the stopping rule.

**One boosting round, the full loop**
1. p = sigmoid(z); g = p-y; h = p(1-p) for every point, using the CURRENT z
2. find the best split via gain; compute leaf_value for each resulting leaf
3. z_new = z + learning_rate * tree_output
4. repeat for n_estimators rounds, recomputing g and h from the updated z each time


In [ ]:
import numpy as np

# same 4-point toy set used in the hand-derivation: perfectly separable by x, one feature
X = np.array([1, 2, 8, 9], dtype=float).reshape(-1, 1)
y = np.array([0, 0, 1, 1], dtype=float)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient(p, y):   # g = p - y
    return p - y

def hessian(p):        # h = p(1-p), same expression as sigmoid's own derivative
    return p * (1 - p)

In [ ]:
# leaf_score plays gini's role: scores one group. leaf_value is the optimal correction for it.
def leaf_score(g, h, lam=0.0):
    return (np.sum(g) ** 2) / (np.sum(h) + lam)

def leaf_value(g, h, lam=0.0):
    return -np.sum(g) / (np.sum(h) + lam)

In [ ]:
# same search structure as notebook 2's best_split, but gain (from g/h) replaces weighted_gini,
# and we now maximize instead of minimize
def best_split_xgb(X, g, h, lam=0.0):
    best_gain = -float("inf")
    best_feature, best_threshold = None, None
    parent_score = leaf_score(g, h, lam)

    for feature_idx in range(X.shape[1]):
        values = np.unique(X[:, feature_idx])
        thresholds = (values[:-1] + values[1:]) / 2

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            g_left, h_left = g[left_mask], h[left_mask]
            g_right, h_right = g[~left_mask], h[~left_mask]

            if len(g_left) == 0 or len(g_right) == 0:
                continue

            gain = leaf_score(g_left, h_left, lam) + leaf_score(g_right, h_right, lam) - parent_score

            if gain > best_gain:
                best_gain = gain
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gain

In [ ]:
# one full round: current z -> g,h -> best split -> leaf values -> updated z
def boosting_round(X, y, z, lam=0.0, lr=0.3):
    p = sigmoid(z)
    g = gradient(p, y)
    h = hessian(p)

    feature, threshold, gain = best_split_xgb(X, g, h, lam)

    left_mask = X[:, feature] <= threshold
    left_value = leaf_value(g[left_mask], h[left_mask], lam)
    right_value = leaf_value(g[~left_mask], h[~left_mask], lam)

    tree_output = np.where(left_mask, left_value, right_value)
    z_new = z + lr * tree_output

    return z_new, feature, threshold, gain, left_value, right_value

In [ ]:
z = np.zeros(4)   # start at z=0 -> p=0.5 for everyone, same as the hand-derivation

for round_num in range(1, 3):
    z, feature, threshold, gain, left_val, right_val = boosting_round(X, y, z, lam=0.0, lr=0.3)
    p = sigmoid(z)
    print(f"round {round_num}: split=feature{feature}<={threshold}, gain={gain:.3f}, left_val={left_val:.3f}, right_val={right_val:.3f}")
    print("  p:", np.round(p, 3))

# expected, from the hand-derivation: round1 p=[0.354,0.354,0.646,0.646], round2 p=[0.257,0.257,0.743,0.743]

In [ ]:
from xgboost import XGBClassifier

# approximate check against the real library on the same toy data. won't match to the decimal
# (xgboost's real split-finding/internals differ in minor ways) but direction and scale should agree
xgb_toy = XGBClassifier(n_estimators=2, max_depth=1, learning_rate=0.3, reg_lambda=0,
                          base_score=0.5, eval_metric="logloss")
xgb_toy.fit(X, y)
print("xgboost p:", np.round(xgb_toy.predict_proba(X)[:, 1], 3))

### Likely interview questions

**Q: Key difference between plain gradient boosting and XGBoost?**
XGBoost uses second-order info (the Hessian) alongside the gradient — Newton's method instead of plain gradient descent — giving a directly computed near-optimal leaf value instead of just a direction. Regularization (λ, γ) is built into the split/leaf formulas, and splits are scored by actual loss reduction (gain), not a generic purity measure.

**Q: What do λ and γ do?**
λ regularizes leaf values — shrinks -Σg/(Σh+λ), especially for leaves with few points (small Σh), stopping tiny leaves from producing extreme corrections. γ is a fixed penalty per split — a split is only taken if its gain exceeds γ, which stops the tree growing splits that barely help.

**Q: Why gain instead of Gini for splitting?**
Gain is computed directly from the actual loss being optimized (via g and h), so it measures exactly how much a split reduces loss. Gini is a generic impurity measure — a reasonable proxy for classification, but not exact, and it doesn't generalize to other objectives (regression, ranking) the way gradient/Hessian does.

**Q: What does scale_pos_weight do mechanically?**
Multiplies the gradient and Hessian of positive-class (fraud) examples by that factor, so their contribution to every gain and leaf-value calculation counts proportionally more — same imbalance idea as class_weight in logistic regression, inside a different mechanism.

**Q: Why does gain shrink round over round?**
As predictions improve, p moves closer to y, so g=p-y shrinks toward zero. Smaller gradients need smaller corrections — gain decreasing is a built-in convergence signal, not a bug.

**Q: Is each individual tree solving classification or regression?**
Regression — every tree predicts a continuous value (a piece of the running log-odds score), never a class directly. Only the final summed score, passed through sigmoid, becomes a probability.